In [ ]:
# October 30 sample; replace these two explicit paths with your prepared data and selection.
# The entrypoint uses models/<MODEL_ID>/parameters.json from this checkout.
MODEL_ID = "usecase2"
RUN_ID = "cd97jkwzjyqo"
TRAINING_MODE = "1"  # 1: preliminary; 2: full.
MODEL_METADATA_URI = "none"  # Full training requires the preliminary model_metadata.json S3 URI.

S3_INPUTS = {
    "prepared_data": f"s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/{MODEL_ID}/runs/{RUN_ID}/data/prepared/data.csv",
    "selection": f"s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/{MODEL_ID}/runs/{RUN_ID}/data/selection/selected_features.csv",
}


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from urllib.parse import urlsplit
import importlib
import json
import os
import sys

import boto3
import pandas as pd
from botocore.config import Config as ClientConfig
from IPython import get_ipython
from IPython.display import display

# Reload edited model modules as well as the entrypoint on the next cell execution.
get_ipython().run_line_magic("load_ext", "autoreload")
get_ipython().run_line_magic("autoreload", "2")

REPOSITORY = next(p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "mlops-models/src/mlops_models").is_dir())
sys.path.insert(0, str(REPOSITORY / "mlops-models/src"))
parameter_path = REPOSITORY / "mlops-models/src/mlops_models/models" / MODEL_ID / "parameters.json"

work_directory = Path(mkdtemp(prefix="train-"))
input_directory = work_directory / "input"
data_directory = input_directory / "prepared"  # train.execute appends /prepared to input_path.
selection_file = work_directory / "selection/selected_features.csv"
output_directory = work_directory / "output"
output_directory.mkdir()
local_files = {"prepared_data": data_directory / "data.csv", "selection": selection_file}

os.environ.update({
    "MODEL_ID": MODEL_ID,
    "MLOPS_TRAINING_MODE": TRAINING_MODE,
    "MLOPS_MODEL_METADATA_URI": MODEL_METADATA_URI,
    "MLOPS_TRACKING_URI": "",
    "MLOPS_WRITE_RUN_REPORT": "0",
    "MLOPS_LOG_ARTIFACTS": "0",
})
s3 = boto3.client("s3", config=ClientConfig(
    connect_timeout=5, read_timeout=15, retries={"total_max_attempts": 2}))
for name, local_file in local_files.items():
    address = urlsplit(S3_INPUTS[name])
    if address.scheme != "s3" or not address.netloc or not address.path.strip("/"):
        raise ValueError(f"Set S3_INPUTS[{name!r}] to an S3 object path in the first cell.")
    local_file.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {S3_INPUTS[name]} -> {local_file}", flush=True)
    s3.download_file(address.netloc, address.path.lstrip("/"), str(local_file))
    assert local_file.is_file(), f"Input was not downloaded: {local_file}"
print("Parameters used by the entrypoint:", parameter_path)
print("Local output directory:", output_directory)


In [ ]:
# Optional inspection; execution does not depend on this cell.
display(json.loads(parameter_path.read_text()))
display(pd.read_csv(selection_file))
display(pd.read_csv(data_directory / "data.csv", nrows=10, dtype=str))


In [ ]:
from mlops_models.entrypoints import train

# Keep each fit's outputs separate when rerunning this cell.
run_directory = Path(mkdtemp(prefix="fit-", dir=output_directory))
model_directory = run_directory / "model"
report_directory = run_directory / "reports"
model_directory.mkdir()
report_directory.mkdir()
os.environ.update({
    "MLOPS_MODEL_DIR": str(model_directory),
    "SM_MODEL_DIR": str(model_directory),
    "SM_OUTPUT_DATA_DIR": str(report_directory),
    "SM_OUTPUT_DIR": str(report_directory),
})
train = importlib.reload(train)
print("Entrypoint:", train.__file__)
result_directory = train.execute(
    model_id=MODEL_ID,
    input_path=input_directory,
    model_dir=model_directory,
    output_dir=report_directory,
    selection_path=selection_file,
)
print("Model directory:", model_directory)
print("Report directory:", report_directory)
result_directory
